In [2]:
import pandas as pd
import json

# 1. Load the data from the data subfolder
with open('../data/raw_credit_applications.json') as f:
    data = json.load(f)

# 2. Flatten the JSON into a table
df = pd.json_normalize(data)

# 3. FIX: Convert lists/dicts to strings so we can check for duplicates
df_fixed = df.astype(str)

# 4. Hunt for "Intentional Issues" (Point 10.1)
print("--- DATA QUALITY AUDIT ---")
print(f"Total Applications: {len(df)}")
print(f"Duplicates found: {df_fixed.duplicated().sum()}") # Now this will work!

print("\n--- Missing Values ---")
print(df.isnull().sum())

--- DATA QUALITY AUDIT ---
Total Applications: 502
Duplicates found: 0

--- Missing Values ---
_id                                   0
spending_behavior                     0
processing_timestamp                440
applicant_info.full_name              0
applicant_info.email                  0
applicant_info.ssn                    5
applicant_info.ip_address             5
applicant_info.gender                 1
applicant_info.date_of_birth          1
applicant_info.zip_code               1
financials.annual_income              5
financials.credit_history_months      0
financials.debt_to_income             0
financials.savings_balance            0
decision.loan_approved                0
decision.rejection_reason           292
loan_purpose                        452
decision.interest_rate              210
decision.approved_amount            210
financials.annual_salary            497
notes                               500
dtype: int64


In [ ]:
# 1. Create a cleaned version of the data
# We remove rows where critical information (SSN or Income) is missing
# (Based on your audit, this should remove about 5-10 broken records)
df_cleaned = df.dropna(subset=['applicant_info.ssn', 'financials.annual_income'])

# 2. Summary of the cleaning process
print(f"--- CLEANING SUMMARY ---")
print(f"Original records: {len(df)}")
print(f"Cleaned records: {len(df_cleaned)}")
print(f"Removed {len(df) - len(df_cleaned)} records due to missing critical information.")

# 3. Save the cleaned data as a CSV for your teammates 
# This file will appear in your 'data' folder
df_cleaned.to_csv('../data/cleaned_credit_data.csv', index=False)
print("\nSuccess: Cleaned data saved to '../data/cleaned_credit_data.csv'")

--- CLEANING SUMMARY ---
Original records: 502
Cleaned records: 492
Removed 10 records due to missing critical information.

Success: Cleaned data saved to '../data/cleaned_credit_data.csv'


Data Quality Report - Team 11

Initial State: 502 raw credit applications loaded from JSON.

Issues Identified: Identified 5 missing SSNs, 5 missing annual income values, and significant missingness in timestamps and notes.

Remediation: Removed 10 records with missing critical identifiers to ensure model accuracy.

Final Dataset: 492 valid records exported to cleaned_credit_data.csv

### Data Quality Audit - Dimensions Mapping
The following issues were identified during the initial audit and are mapped to standard Data Quality dimensions.

| Issue Found | Dimension | Impact |
| :--- | :--- | :--- |
| 5 Missing SSNs | **Completeness** | Critical: Cannot identify applicants. |
| 5 Missing Annual Incomes | **Completeness** | Critical: Cannot calculate credit risk. |
| 440 Missing Timestamps | **Completeness** | Moderate: Affects audit trail. |
| Inconsistent Gender Coding | **Consistency** | High: Affects Bias Analysis. |
| Negative Credit History | **Validity** | High: Impossible value. |

In [4]:
# 1. Check for Inconsistent Gender Coding (e.g., 'M', 'm', 'Male')
print("--- Gender Coding Consistency Check ---")
print(df['applicant_info.gender'].unique())

# 2. Check for Invalid Values (Negative months of credit history)
invalid_months = df[df['financials.credit_history_months'] < 0]
print(f"\n--- Invalid Values Check ---")
print(f"Number of records with negative credit history: {len(invalid_months)}")

# 3. Check for Data Type Mismatches (e.g., income as string)
print(f"\n--- Data Type Check ---")
print(f"Income column type: {df['financials.annual_income'].dtype}")

--- Gender Coding Consistency Check ---
['Male' 'M' 'F' 'Female' '' nan]

--- Invalid Values Check ---
Number of records with negative credit history: 2

--- Data Type Check ---
Income column type: object


In [5]:
# Final Remediation (Step 11.1)
# 1. Convert Income to numeric
df_cleaned['financials.annual_income'] = pd.to_numeric(df_cleaned['financials.annual_income'], errors='coerce')

# 2. Fix Gender Coding (Standardizing to 'Male' and 'Female')
df_cleaned['applicant_info.gender'] = df_cleaned['applicant_info.gender'].replace({'M': 'Male', 'F': 'Female'})

# 3. Remove negative credit history months
df_cleaned = df_cleaned[df_cleaned['financials.credit_history_months'] >= 0]

# Overwrite the cleaned file with the final version
df_cleaned.to_csv('../data/cleaned_credit_data.csv', index=False)
print("Final cleaned data exported with standardized genders and valid credit history.")

Final cleaned data exported with standardized genders and valid credit history.


/var/folders/2z/3b8g43wx5ts0gvxpcpcw2khm0000gn/T/ipykernel_45352/4206971287.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['financials.annual_income'] = pd.to_numeric(df_cleaned['financials.annual_income'], errors='coerce')
/var/folders/2z/3b8g43wx5ts0gvxpcpcw2khm0000gn/T/ipykernel_45352/4206971287.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['applicant_info.gender'] = df_cleaned['applicant_info.gender'].replace({'M': 'Male', 'F': 'Female'})
